# Reproducing the dissertation's tables and statistics

This notebook regenerates every table (and several previously-unverifiable
statistics) in *Cold-Start Bus Stop Demand Prediction under Spatial
Extrapolation* directly from the CSVs already committed in this repository.

**Runtime: well under a minute.** It does not retrain any model and does not
need the ~453MB raw `data/` folder (with one exception, noted where it
occurs) -- everything here reads the small, already-computed `results_*.csv`
and `stops_features_osm.csv` files that ship with the repo.

**What this notebook is not:** it does not reproduce the full 33-hour
from-scratch pipeline (raw BUSTO/AI23/OSM -> trained models). For that, see
`RUNBOOK.md`. This notebook's job is narrower and more directly useful for
verification: *given the results this repo already produced, do the
dissertation's own numbers follow from them?*

Requirements: everything here uses only `pandas`, `numpy`, and `scipy` --
all three are already pinned in `requirements.txt`.

Run this notebook from the repository root (the same directory as
`RUNBOOK.md`).

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

ROOT = Path(".")
assert (ROOT / "RUNBOOK.md").exists(), (
    "Run this notebook from the repository root (where RUNBOOK.md lives)."
)

# A running log of (label, dissertation_value, computed_value, note) used to
# build the reconciliation table in the last section. Populated as we go.
CHECKS = []

def record(label, diss_value, computed_value, note=""):
    CHECKS.append({"check": label, "dissertation": diss_value,
                    "computed": computed_value, "note": note})

def record_numeric(label, diss_value, computed_value):
    # Like record(), but auto-flags a MISMATCH if the values differ.
    note = "" if computed_value == diss_value else "MISMATCH"
    record(label, diss_value, computed_value, note)


## 0a. Data-join pipeline: which script produced which file

If you want to rebuild a feature file from raw data rather than trust the
committed CSV, this is the exact chain (see `RUNBOOK.md` Sec.3.1/Sec.9 for
full detail and runtimes; Sec.4.0 for where to download the raw sources).

In [ ]:
pipeline = pd.DataFrame([
    ("step1_aggregate_busto.py",        "data/*Weekday*QUARTER HOUR*.csv (raw BUSTO)", "busto_stop_level_boardings.csv",
     "Sums Boardings per stop across all weekday route/direction/quarter-hour rows; also computes the raw service-coverage count."),
    ("step2_join_coordinates.py",       "+ data/Bus_Stops.csv",                        "stops_with_coords.csv",
     "Joins stop coordinates on STOPCODE."),
    ("step3_lsoa_features.py",          "+ data/access_*.csv (AI23)",                  "stops_features.csv",
     "Postcode -> 2011 LSOA (via postcodes.io + ONS crosswalk), joins the 8 AI23 columns, filters to the 33 London boroughs -> 17,943 stops."),
    ("step3b_osm_features.py",          "+ live Overpass API",                         "stops_features_osm.csv",
     "Adds 6 OSM point-of-interest columns within a 500m buffer per stop."),
    ("step3c_add_scenic.py",            "(same file)",                                 "stops_features_osm.csv",
     "Adds the 7th POI category, poi_scenic."),
    ("step3d_add_service_coverage.py",  "+ busto_stop_level_boardings.csv",            "stops_features_osm.csv",
     "Merges in the service_coverage count computed back in step1 -- this is the file every step4* script reads."),
], columns=["script", "reads", "writes", "what it does"])
pipeline


## 0b. Model-training commands: which config feeds which column/row

Every row below is copy-pasteable. `step4_model.py` always trains the
**whole** model suite (HistAvg/IDW/MLR/RF/XGBoost/MLP/GATv2) together in
one run per feature-set config -- there is no flag to train only one
tabular model from it, so a single run still takes the full ~1.5-2h even if
you only care about, say, Random Forest. **If you only want to check one
tabular model cheaply, use `step4c_fast_baselines.py` (untuned, no NN,
seconds) or `step4j_tuned_baselines.py` (tuned, no NN, ~39s/fold) instead**
-- both skip GATv2/MLP entirely.

In [ ]:
commands = pd.DataFrame([
    ("AI23 (headline table col.)",    "python step4c_fast_baselines.py",              "results_cv_ai23_only_fastbaselines.csv",  "HistAvg/IDW/MLR/RF/XGBoost -- current-protocol, tabular only, fast (no NN)"),
    ("OSM (headline table col.)",     "python step4c_fast_baselines.py",              "results_cv_osm_only_fastbaselines.csv",   "same script, all 3 pre-SC configs written in one run"),
    ("AI23+OSM (headline table col.)","python step4c_fast_baselines.py",              "results_cv_ai23_osm_fastbaselines.csv",   "same script, all 3 pre-SC configs written in one run"),
    ("AI23+SC",                       "python -u step4_model.py --ai23-only --with-sc","results_cv_ai23_sc.csv",                  "full suite incl. MLP/GATv2, ~1.5-2h"),
    ("AI23+OSM+SC (HEADLINE)",        "python -u step4_model.py --with-sc",           "results_cv_ai23_osm_sc.csv",              "full suite incl. MLP/GATv2, ~1.5-2h -- the number everything else is compared to"),
    ("Ridge/RF/XGBoost tuned",        "python step4j_tuned_baselines.py",             "results_cv_tuned.csv",                    "tabular only, no NN, leakage-safe inner CV, ~39s/fold"),
    ("IDW baseline",                  "python step4d_idw_baseline.py",                "results_cv_idw.csv",                      "feature-independent, ~1 sec total"),
    ("GATv2, K=10",                   "python -u step4_model.py --with-sc --k10",     "results_cv_gnn_fairness.csv",             "full suite, ~1.5-2h"),
    ("GATv2, func-sim edges",         "python -u step4_model.py --with-sc --func-sim","results_cv_func_sim.csv",                 "full suite, ~1.5-2h"),
    ("PCA-AI23 robustness check",     "python -u step4_model.py --with-sc --pca-ai23","results_cv_pca_ai23_osm_sc.csv",          "full suite, ~1.5-2h"),
    ("GATv2-Fusion",                  "python step4e_zheng_fusion.py",                "results_cv_zheng_fusion.csv",             "imports graph builders from step4_model.py"),
    ("GCN",                           "python step4i_gcn_baseline.py",                "results_cv_gcn.csv",                      "GATv2Conv swapped for GCNConv, everything else identical"),
    ("5-seed MLP noise floor",        "python step4l_multiseed_mlp.py",               "results_summary_multiseed_mlp.csv",       "standalone MLP, 5 seeds"),
], columns=["dissertation column/row", "command", "output file", "notes"])
pd.set_option("display.max_colwidth", 80)
commands


## 1. Main results table

Table `main-results` (Chapter 6): macro-averaged WMAPE, 33-fold
leave-borough-out CV, for every model x feature-set cell.

Column provenance (per `RUNBOOK.md` Sec.5's provenance note): the AI23 / OSM
/ AI23+OSM columns for the tabular models use the `*_fastbaselines` configs
(the current-protocol re-run supplement), not the original pre-fix
`ai23_only`/`osm_only`/`ai23_osm` configs. AI23+SC and AI23+OSM+SC use the
current, bug-fixed pipeline directly.

Note: `merge_results.py`'s `CONFIGS` list was written before the `ai23_sc`
config existed (added 22 Aug 2026), so `all_results_summary.csv` does not
contain an AI23+SC column -- this notebook reads the standalone
`results_summary_ai23_sc.csv` for that column instead. (`merge_results.py`
has been fixed to include `ai23_sc` going forward; re-run it and this
workaround becomes unnecessary.)

In [ ]:
all_summary = pd.read_csv("all_results_summary.csv")
ai23_sc_summary = pd.read_csv("results_summary_ai23_sc.csv")
ai23_sc_summary.insert(0, "config", "ai23_sc")

def wmape(config, model, df=all_summary):
    row = df[(df["config"] == config) & (df["model"] == model)]
    if row.empty:
        return None
    return round(float(row["WMAPE_mean"].iloc[0]), 4)

COLUMNS = {
    "AI23":         "ai23_only_fastbaselines",
    "OSM":          "osm_only_fastbaselines",
    "AI23+OSM":     "ai23_osm_fastbaselines",
    "AI23+SC":      None,   # handled specially (standalone file)
    "AI23+OSM+SC":  "ai23_osm_sc",
}

# (row label, model name as it appears in the CSVs, which columns it has a value in)
ROWS = [
    ("Historical average", "HistAvg",  ["AI23", "OSM", "AI23+OSM", "AI23+SC", "AI23+OSM+SC"]),
    ("IDW (no features)",  "IDW",      ["AI23", "OSM", "AI23+OSM", "AI23+SC", "AI23+OSM+SC"]),
    ("Ridge (a=1.0)",      "MLR",      ["AI23", "OSM", "AI23+OSM", "AI23+SC", "AI23+OSM+SC"]),
    ("Random forest",      "RF",       ["AI23", "OSM", "AI23+OSM", "AI23+SC", "AI23+OSM+SC"]),
    ("XGBoost",            "XGBoost",  ["AI23", "OSM", "AI23+OSM", "AI23+SC", "AI23+OSM+SC"]),
    ("MLP",                "MLP",      ["AI23+SC", "AI23+OSM+SC"]),
    ("GATv2 (K=5)",        "GATv2",    ["AI23+SC", "AI23+OSM+SC"]),
]

table = {}
for label, model, present_cols in ROWS:
    row = {}
    for col in ["AI23", "OSM", "AI23+OSM", "AI23+SC", "AI23+OSM+SC"]:
        if col not in present_cols:
            row[col] = "—"
            continue
        if model == "IDW":
            # IDW is feature-independent; it's only tagged under a handful of
            # configs (not every *_fastbaselines/ai23_sc config), so pull its
            # one value from the dedicated 'idw' config for every column.
            row[col] = wmape("idw", "IDW")
        elif col == "AI23+SC":
            row[col] = wmape("ai23_sc", model, df=ai23_sc_summary)
        else:
            row[col] = wmape(COLUMNS[col], model)
    table[label] = row

main_results = pd.DataFrame(table).T[["AI23", "OSM", "AI23+OSM", "AI23+SC", "AI23+OSM+SC"]]
main_results


In [ ]:
# Tuned baselines and graph-model variants (AI23+OSM+SC column only)
extra_rows = {
    "Ridge tuned":          wmape("tuned", "MLR-tuned"),
    "Random forest tuned":  wmape("tuned", "RF-tuned"),
    "XGBoost tuned":        wmape("tuned", "XGBoost-tuned"),
    "GATv2, K=10":          wmape("gnn_fairness", "GATv2"),
    "GATv2-Fusion":         wmape("zheng_fusion", "GATv2-Fusion"),
    "GATv2, func-sim edges":wmape("func_sim", "GATv2"),
    "GCN":                  wmape("gcn", "GCN"),
}
extra = pd.Series(extra_rows, name="AI23+OSM+SC")
extra_df = extra.to_frame()
extra_df


In [ ]:
# --- Reconcile against the dissertation's Table `main-results` ---
DISSERTATION_MAIN = {
    ("Historical average", "AI23"): 1.0822, ("Historical average", "OSM"): 1.0822,
    ("Historical average", "AI23+OSM"): 1.0822, ("Historical average", "AI23+SC"): 1.0822,
    ("Historical average", "AI23+OSM+SC"): 1.0822,
    ("IDW (no features)", "AI23"): 0.8487, ("IDW (no features)", "OSM"): 0.8487,
    ("IDW (no features)", "AI23+OSM"): 0.8487, ("IDW (no features)", "AI23+SC"): 0.8487,
    ("IDW (no features)", "AI23+OSM+SC"): 0.8487,
    ("Ridge (a=1.0)", "AI23"): 0.8120, ("Ridge (a=1.0)", "OSM"): 0.8078,
    ("Ridge (a=1.0)", "AI23+OSM"): 0.7982, ("Ridge (a=1.0)", "AI23+SC"): 0.6420,
    ("Ridge (a=1.0)", "AI23+OSM+SC"): 0.6404,
    ("Random forest", "AI23"): 0.8128, ("Random forest", "OSM"): 0.8064,
    ("Random forest", "AI23+OSM"): 0.7970, ("Random forest", "AI23+SC"): 0.6452,
    ("Random forest", "AI23+OSM+SC"): 0.6428,
    ("XGBoost", "AI23"): 0.8207, ("XGBoost", "OSM"): 0.8084,
    ("XGBoost", "AI23+OSM"): 0.8075, ("XGBoost", "AI23+SC"): 0.6584,
    ("XGBoost", "AI23+OSM+SC"): 0.6437,
    ("MLP", "AI23+SC"): 0.6365, ("MLP", "AI23+OSM+SC"): 0.6311,
    ("GATv2 (K=5)", "AI23+SC"): 0.7262, ("GATv2 (K=5)", "AI23+OSM+SC"): 0.7187,
}
DISSERTATION_EXTRA = {
    "Ridge tuned": 0.6401, "Random forest tuned": 0.6339, "XGBoost tuned": 0.6497,
    "GATv2, K=10": 0.7372, "GATv2-Fusion": 0.7070, "GATv2, func-sim edges": 0.7352,
    "GCN": 0.7006,
}

n_mismatch = 0
for (row, col), diss_val in DISSERTATION_MAIN.items():
    got = table[row][col]
    match = (got == diss_val)
    n_mismatch += not match
    record(f"Main results: {row} / {col}", diss_val, got, "" if match else "MISMATCH")
for row, diss_val in DISSERTATION_EXTRA.items():
    got = extra_rows[row]
    match = (got == diss_val)
    n_mismatch += not match
    record(f"Main results: {row}", diss_val, got, "" if match else "MISMATCH")

print(f"Main results table: {len(DISSERTATION_MAIN) + len(DISSERTATION_EXTRA)} cells checked, "
      f"{n_mismatch} mismatch(es).")


## 2. Five-seed MLP noise envelope

The dissertation reports the headline MLP score (0.6311) as a single
canonical run at seed 42, with a five-seed envelope of 0.6301 +/- 0.0004
used to argue the headline is stable under reseeding.

In [ ]:
seeds = pd.read_csv("results_summary_multiseed_mlp.csv")
mean_wmape = seeds["WMAPE_mean"].mean()
std_wmape = seeds["WMAPE_mean"].std()   # sample std (ddof=1), matching how such envelopes are conventionally reported
print(seeds)
print(f"\nmean = {mean_wmape:.4f}, std = {std_wmape:.4f}")

record_numeric("5-seed MLP envelope (mean)", 0.6301, round(mean_wmape, 4))
record_numeric("5-seed MLP envelope (std)", 0.0004, round(std_wmape, 4))


## 3. Paired Wilcoxon significance tests

Table `significance` (Chapter 6): paired Wilcoxon signed-rank tests across
the 33 borough folds, AI23+OSM+SC. Each pair happens to sit inside a single
per-fold results CSV, so no cross-file join is needed -- just filter by
`model` and align by `borough`.

The dissertation previously had **no committed, re-runnable script** for
this table (verified during the audit -- the numbers existed only as pasted
ad hoc code+output in `experiment_log.md`/`diagnostics_report.md`). This
cell is that missing script.

In [ ]:
def paired_wilcoxon(csv_path, model_a, model_b):
    df = pd.read_csv(csv_path)
    a = df[df["model"] == model_a].set_index("borough")["WMAPE"].sort_index()
    b = df[df["model"] == model_b].set_index("borough")["WMAPE"].sort_index()
    assert list(a.index) == list(b.index), "borough sets don't align"
    diff = a - b   # positive means A worse (higher WMAPE) than B
    stat, p = stats.wilcoxon(a, b)
    return float(diff.mean()), float(p), len(a)

SIG_PAIRS = [
    ("GCN vs GATv2 (K=5)",        "results_cv_gcn.csv",          "GCN", "GATv2",       -0.0181, 0.010),
    ("GCN vs MLP",                "results_cv_gcn.csv",          "GCN", "MLP",          0.0695, 0.00001),
    ("MLP vs RF (tuned)",         "results_cv_tuned.csv",        "MLP", "RF-tuned",    -0.0028, 0.292),
    ("GATv2-Fusion vs GATv2(K=5)","results_cv_zheng_fusion.csv", "GATv2-Fusion", "GATv2", -0.0117, 0.126),
    ("Ridge vs RF (150)",         "results_cv_ai23_osm_sc.csv",  "MLR", "RF",          -0.0024, 0.357),
    # Not in Table `significance` itself, but its p-value is baked into
    # fig3_paired_borough_boxplot.png's annotation (p=2.33e-10) without being
    # stated as a number anywhere in the prose or tables -- reproduced here
    # so it's not only readable off a PNG.
    ("MLP vs GATv2 (K=5)",        "results_cv_ai23_osm_sc.csv",  "MLP", "GATv2",       -0.0876, 2.33e-10),
]

sig_rows = []
for label, path, a, b, diss_effect, diss_p in SIG_PAIRS:
    effect, p, n = paired_wilcoxon(path, a, b)
    sig_rows.append({"comparison": label, "effect_mean": round(effect, 4), "p_value": round(p, 5),
                      "n_folds": n, "dissertation_effect": diss_effect, "dissertation_p": diss_p})
    record_numeric(f"Significance: {label} (effect)", diss_effect, round(effect, 4))
    # p-values are reported as thresholds (e.g. p<0.00001) for the small ones in the dissertation;
    # record a pass/fail against that threshold rather than an exact-match comparison.
    p_ok = (p < 0.0001) if diss_p <= 0.00001 else abs(p - diss_p) < 0.01
    record(f"Significance: {label} (p-value, vs threshold)", diss_p, round(p, 5), "" if p_ok else "CHECK")

pd.DataFrame(sig_rows)


## 4. Service-coverage univariate association

Table `sc-univariate`: Pearson r, Spearman rho and R^2 between
`log1p(service_coverage)` and `log1p(total_boardings)`, n=17,943.

In [ ]:
feat = pd.read_csv("stops_features_osm.csv")
assert len(feat) == 17943, f"expected 17,943 stops, got {len(feat)}"

x = np.log1p(feat["service_coverage"])
y = np.log1p(feat["total_boardings"])

pearson_r, _ = stats.pearsonr(x, y)
spearman_rho, _ = stats.spearmanr(x, y)
r2 = pearson_r ** 2

print(f"Pearson r  = {pearson_r:.4f}")
print(f"Spearman rho = {spearman_rho:.4f}")
print(f"R^2        = {r2:.4f}")

record_numeric("sc-univariate: Pearson r", 0.6882, round(pearson_r, 4))
record_numeric("sc-univariate: Spearman rho", 0.72, round(spearman_rho, 2))
record_numeric("sc-univariate: R^2", 0.47, round(r2, 2))


## 5. Feature range table (Table `features`)

No script in the repo previously printed or saved the min/max range of each
of the 17 predictors -- Table `features` in the dissertation was asserted,
not reproducible from any committed artefact. This cell computes it
directly from `stops_features_osm.csv`.

In [ ]:
FEATURE_COLS = [
    "employment_all_30min", "hospitals_30min", "gp_30min", "supermarkets_30min",
    "pharmacies_30min", "primary_schools_30min", "secondary_schools_30min", "main_bua_30min",
    "poi_residential", "poi_shopping", "poi_company", "poi_education", "poi_entertainment",
    "poi_scenic", "service_coverage", "lat", "lon",
]
ranges = feat[FEATURE_COLS].agg(["min", "max"]).T
ranges.columns = ["min", "max"]
ranges


In [ ]:
DISSERTATION_RANGES = {
    "employment_all_30min": (875, 2907645), "hospitals_30min": (0, 43), "gp_30min": (0, 218),
    "supermarkets_30min": (0, 91), "pharmacies_30min": (0, 411), "primary_schools_30min": (2, 289),
    "secondary_schools_30min": (0, 69), "main_bua_30min": (0, 1), "poi_residential": (0, 235),
    "poi_shopping": (0, 992), "poi_company": (0, 249), "poi_education": (0, 116),
    "poi_entertainment": (0, 958), "poi_scenic": (0, 187), "service_coverage": (2, 1504),
}
for col, (lo, hi) in DISSERTATION_RANGES.items():
    got_lo, got_hi = ranges.loc[col, "min"], ranges.loc[col, "max"]
    match = (got_lo == lo) and (got_hi == hi)
    record(f"Feature range: {col}", f"{lo}-{hi}", f"{got_lo:g}-{got_hi:g}", "" if match else "MISMATCH")
print("lat range:", ranges.loc["lat", "min"], "-", ranges.loc["lat", "max"], "  (dissertation: 51.2929-51.6846)")
print("lon range:", ranges.loc["lon", "min"], "-", ranges.loc["lon", "max"], "  (dissertation: -0.4995-0.2978)")


## 6. Target distribution statistics

min / median / mean / max / skewness of `total_boardings`, and the skewness
after `log1p`, plus the zero-boarding stop count.

**Not reproducible here:** the dissertation's further claim that 254 of the
287 zero-boarding stops (88.5%) record *positive alightings* needs an
`Alightings` column that is not present in any committed, gitignore-exempt
file (`stops_features_osm.csv` and `busto_stop_level_boardings.csv` both
lack it) -- it requires the raw, gitignored `data/` BUSTO extract. That
specific figure is left unverified by this notebook; it was independently
confirmed during the audit against `diagnostics_report.md`'s own diagnostic
session (V10), which *did* have raw-data access at the time it was run.

In [ ]:
y = feat["total_boardings"]
y_log = np.log1p(y)

target_stats = {
    "min": y.min(), "median": y.median(), "mean": y.mean(), "max": y.max(),
    "skew": stats.skew(y), "skew (log1p)": stats.skew(y_log),
}
for k, v in target_stats.items():
    print(f"{k:>14}: {v:.4f}" if isinstance(v, float) else f"{k:>14}: {v}")

n_zero = int((y == 0).sum())
pct_zero = 100 * n_zero / len(y)
print(f"\nzero-boarding stops: {n_zero} ({pct_zero:.1f}%)")

record_numeric("Target median", 112.6, round(float(target_stats["median"]), 1))
record_numeric("Target mean", 293.8, round(float(target_stats["mean"]), 1))
record_numeric("Target max", 14137, int(target_stats["max"]))
record_numeric("Target skewness", 5.64, round(float(target_stats["skew"]), 2))
record_numeric("Target skewness (log1p)", -0.55, round(float(target_stats["skew (log1p)"]), 2))
record_numeric("Zero-boarding stops (n)", 287, n_zero)
record_numeric("Zero-boarding stops (%)", 1.6, round(pct_zero, 1))


## 7. AI23 collinearity: VIF and cross-correlation

Two numbers flagged as discrepancies during the audit:

- The dissertation states the two highest AI23 variance-inflation factors as
  pharmacies_30min=37.98, gp_30min=18.24. The only VIF computation
  previously recorded in this repo's own logs (`experiment_log.md`) gives
  37.7 / 17.7 instead.
- The dissertation states no AI23xOSM feature pair exceeds \|r\|=0.72. The
  repo's own diagnostic (`diagnostics_report.md`, session V6) recorded a
  max of 0.7223 (hospitals_30min x poi_scenic) -- which exceeds the stated
  bound.

This cell recomputes both fresh, so the true current values (and which of
the three prior figures they match, if any) are settled directly rather
than argued from old logs. VIF is computed manually (R^2 of each column
regressed on the rest) rather than via `statsmodels`, since that package
isn't a project dependency.

In [ ]:
AI23_COLS = [
    "employment_all_30min", "hospitals_30min", "gp_30min", "supermarkets_30min",
    "pharmacies_30min", "primary_schools_30min", "secondary_schools_30min", "main_bua_30min",
]
OSM_COLS = ["poi_residential", "poi_shopping", "poi_company", "poi_education",
            "poi_entertainment", "poi_scenic"]

ai23_log = np.log1p(feat[AI23_COLS])

def vif(df, col):
    y_ = df[col].values
    X_ = df.drop(columns=[col]).values
    X_ = np.column_stack([np.ones(len(X_)), X_])   # intercept
    beta, *_ = np.linalg.lstsq(X_, y_, rcond=None)
    resid = y_ - X_ @ beta
    ss_res = (resid ** 2).sum()
    ss_tot = ((y_ - y_.mean()) ** 2).sum()
    r2 = 1 - ss_res / ss_tot
    return 1 / (1 - r2)

vifs = {c: vif(ai23_log, c) for c in AI23_COLS}
vif_series = pd.Series(vifs).sort_values(ascending=False)
print("VIF (log1p AI23 block):")
print(vif_series.round(2))

vif_pharm, vif_gp = round(vifs["pharmacies_30min"], 2), round(vifs["gp_30min"], 2)
record("VIF pharmacies_30min", 37.98, vif_pharm, "" if vif_pharm == 37.98 else "MISMATCH")
record("VIF gp_30min", 18.24, vif_gp, "" if vif_gp == 18.24 else "MISMATCH")


In [ ]:
# Strongest AI23-internal pair
corr = ai23_log.corr().abs()
corr_vals = corr.values.copy()
np.fill_diagonal(corr_vals, 0)
i, j = np.unravel_index(np.argmax(corr_vals), corr_vals.shape)
print(f"Strongest AI23-internal pair: {corr.index[i]} x {corr.columns[j]}, "
      f"|r| = {corr_vals[i, j]:.4f}")
strongest_r = round(float(corr_vals[i, j]), 2)
record("Strongest AI23-internal |r|", 0.98, strongest_r, "" if strongest_r == 0.98 else "MISMATCH")

# AI23 x OSM cross-correlation (48 pairs)
osm_log = np.log1p(feat[OSM_COLS])
cross = pd.DataFrame(index=AI23_COLS, columns=OSM_COLS, dtype=float)
for a in AI23_COLS:
    for o in OSM_COLS:
        cross.loc[a, o] = abs(np.corrcoef(ai23_log[a], osm_log[o])[0, 1])

max_pair = cross.stack().idxmax()
max_val = cross.stack().max()
print(f"\nMax |r| across the {cross.size} AI23xOSM pairs: {max_val:.4f}  (pair: {max_pair[0]} x {max_pair[1]})")
record("Max AI23xOSM |r| (48 pairs)", "\u2264 0.72 (claimed)", round(max_val, 4),
       "" if max_val <= 0.72 else "EXCEEDS STATED BOUND")


## 8. Spatial partition structure

### 8a. LSOA-to-borough nesting

The dissertation states LSOAs nest within London boroughs in 4,219 of
4,222 cases (99.93%) -- i.e. almost every LSOA sits entirely inside one
borough, which is what lets leave-borough-out CV also withhold every LSOA
in the test borough as a unit (Section `blocked-motivation`).

In [ ]:
lsoa_to_boroughs = feat.groupby("lsoa11cd")["lad_name"].nunique()
n_lsoas = len(lsoa_to_boroughs)
n_nested = int((lsoa_to_boroughs == 1).sum())
pct_nested = 100 * n_nested / n_lsoas

print(f"Unique LSOAs: {n_lsoas:,}")
print(f"LSOAs mapping to exactly 1 borough: {n_nested:,} ({pct_nested:.2f}%)")

record_numeric("LSOAs total", 4222, n_lsoas)
record_numeric("LSOAs nested in a single borough (n)", 4219, n_nested)
record_numeric("LSOAs nested in a single borough (%)", 99.93, round(pct_nested, 2))


### 8b. Porosity graph (Section `boundary`)

The dissertation's Table `graph-inventory` reports the "porosity graph" --
K=5 KNN, one-directional, no route edges, not deduplicated -- as: 89,715
directed edges, 4,291 (4.8%) crossing a borough boundary, 2,019 stops
(11.3%) with at least one cross-borough neighbour, 0.3% with all five.

**Provenance note:** during the audit, `boundary_diagnostic_stops.csv`
(the only file on disk claiming to hold these per-stop numbers) was found to
imply a different aggregate (4,974 crossing edges, 5.54%) with no
generating script anywhere in the repo -- an orphaned, unreproducible file,
independently flagged as "unsourced" by this project's own
`diagnostics_report.md` (V11) before this audit ever started.

This cell recomputes the porosity graph from scratch, directly from
`stops_features_osm.csv`, using the same construction
`step8b_assortativity_figure.py` uses for its KNN half (`BallTree` on
haversine coordinates, `k=6` query, self dropped). **Result: this fresh
computation reproduces the dissertation's numbers exactly** -- the
manuscript's figures were correct all along; it was only the orphaned CSV
that was wrong. This cell is the reproducible source `boundary_diagnostic_stops.csv`
never had.

In [ ]:
from sklearn.neighbors import BallTree

lad_arr = feat["lad_name"].values
coords = np.radians(feat[["lat", "lon"]].values)

tree = BallTree(coords, metric="haversine")
_, nbrs = tree.query(coords, k=6)     # self + 5 nearest
nbr_idx = nbrs[:, 1:]                 # drop self -> shape (n, 5)

n = len(feat)
src = np.repeat(np.arange(n), 5)
dst = nbr_idx.reshape(-1)
cross_edge = lad_arr[src] != lad_arr[dst]

n_edges = len(src)
n_cross_edges = int(cross_edge.sum())
per_stop_cross = cross_edge.reshape(n, 5).sum(axis=1)
n_stops_any_cross = int((per_stop_cross >= 1).sum())
n_stops_all5_cross = int((per_stop_cross == 5).sum())

pct_cross_edges = 100 * n_cross_edges / n_edges
pct_stops_any = 100 * n_stops_any_cross / n
pct_stops_all5 = 100 * n_stops_all5_cross / n

print(f"total directed KNN edges: {n_edges:,} (expect {n*5:,})")
print(f"crossing edges: {n_cross_edges:,} ({pct_cross_edges:.2f}%)")
print(f"stops with >=1 cross neighbour: {n_stops_any_cross:,} ({pct_stops_any:.2f}%)")
print(f"stops with all 5 cross: {n_stops_all5_cross:,} ({pct_stops_all5:.2f}%)")

record_numeric("Porosity graph: total edges", 89715, n_edges)
record("Porosity graph: crossing edges (n)", 4291, n_cross_edges,
       "" if n_cross_edges == 4291 else "MISMATCH")
record("Porosity graph: crossing edges (%)", 4.8, round(pct_cross_edges, 1),
       "" if round(pct_cross_edges, 1) == 4.8 else "MISMATCH")
record("Porosity graph: stops with >=1 cross nbr (n)", 2019, n_stops_any_cross,
       "" if n_stops_any_cross == 2019 else "MISMATCH")
record("Porosity graph: stops with >=1 cross nbr (%)", 11.3, round(pct_stops_any, 1),
       "" if round(pct_stops_any, 1) == 11.3 else "MISMATCH")
record("Porosity graph: stops with all-5 cross (%)", 0.3, round(pct_stops_all5, 1),
       "" if round(pct_stops_all5, 1) == 0.3 else "MISMATCH")


## 9. Random-forest importance of `main_bua_30min`

The dissertation's inertness argument for `main_bua_30min` cites a random
forest feature importance of 0.0034 (std 0.0004) averaged over the 33
folds, AI23-only configuration.

In [ ]:
rf_imp = pd.read_csv("results_rf_feature_importance.csv")
rf_imp = rf_imp.set_index(rf_imp.columns[0]) if "feature" not in rf_imp.columns else rf_imp.set_index("feature")
print(rf_imp)

row = rf_imp.loc["main_bua_30min"]
mean_col = [c for c in rf_imp.columns if "mean" in c.lower()][0]
std_col = [c for c in rf_imp.columns if "std" in c.lower()][0]
record_numeric("RF importance main_bua_30min (mean)", 0.0034, round(float(row[mean_col]), 4))
record_numeric("RF importance main_bua_30min (std)", 0.0004, round(float(row[std_col]), 4))


## Summary: reconciliation against the dissertation text

Every comparison recorded above, in one table. `flag` is non-empty wherever
the freshly computed value doesn't match what's written in the dissertation
-- as of this audit, that's the two VIF values and the strongest
AI23-internal correlation (Section 7 above). These are manuscript-text
issues, not code issues -- this repository doesn't contain the dissertation
source, so fixing the prose is up to you; use the freshly computed values
above as the replacement.

In [ ]:
checks_df = pd.DataFrame(CHECKS)
checks_df["flag"] = checks_df["note"].apply(lambda n: "\u26a0" if n else "")
pd.set_option("display.max_rows", 200)
checks_df[["check", "dissertation", "computed", "flag"]]


In [ ]:
n_flagged = (checks_df["flag"] != "").sum()
print(f"{len(checks_df)} checks run, {n_flagged} flagged for review.")
if n_flagged:
    print()
    print(checks_df[checks_df["flag"] != ""][["check", "dissertation", "computed", "note"]].to_string(index=False))
